In [1]:
import tensorflow as tf

print("tf.__version__=", tf.__version__)


tf.__version__= 2.15.0


# 단층 퍼셉트론

In [2]:
import numpy as np

def sigmoid(x): # 시그모이드 함수
    return 1. / (1. + np.exp(-x))

In [3]:
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [22]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name # 게이트 이름
        self.xdata = xdata.reshape(4,2) # 학습 데이터
        self.tdata = tdata.reshape(4,1) # 정답 데이터

        self.W = np.random.rand(self.xdata.shape[1], 1) # 가중치 초기값 설정
        self.b = np.random.rand(1) # 바이어스 초기값 설정
        self.learning_rate = 1e-2 # 학습율

    def loss_func(self):
        delta = 1e-7
        z = np.dot(self.xdata, self.W) + self.b
        y = sigmoid(z) # 활성함수 : 시그모이드 함수 = 이진 분류에서 사용함
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1-y)+delta)) # 이진 분류에 대한 손실값

    def train(self):
        f = lambda x : self.loss_func() # 손실값 계산하는 람다 함수
        print("initial loss value = ", self.loss_func()) # 초기 손실값 계산해서 출력
        for step in range(20001):
            self.W -= self.learning_rate * numerical_derivative(f, self.W) # 가중치 업데이트
            self.b -= self.learning_rate * numerical_derivative(f,self.b) # 바이어스 업데이트
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ",self.loss_func()) # 학습중 손실값 출력

    def predict(self, input_data):
        z = np.dot(input_data, self.W) + self.b # 회귀식 계산
        y = sigmoid(z) # 활성 함수 시그모이드 함수 실행: 0, 1 사이의 확률값 출력
        if y > 0.5:
            result = 1 # 분류값 저장
        else:
            result = 0
        return y, result

    def accuracy(self, test_xdata, test_tdata):
        matched_list = [] # 정답과 일치한 내용 저장
        not_matched_list = [] # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index) # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index) # 정답과 일치한 내용 추가

        accuracy_val = len(matched_list) / len(test_xdata) # 정확도 계산
        return accuracy_val

In [15]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([0,0,0,1]) # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata) # 모델 객체 생성
AND_obj.train() # 학습

initial loss value =  3.7016474958313395
step =  0 loss value =  3.6585283091727527
step =  1000 loss value =  0.9844574932039118
step =  2000 loss value =  0.6507380094857331
step =  3000 loss value =  0.48606236707991207
step =  4000 loss value =  0.3868431734510537
step =  5000 loss value =  0.3205256695976654
step =  6000 loss value =  0.27315908845070347
step =  7000 loss value =  0.2377013200088263
step =  8000 loss value =  0.21020489638658169


In [12]:
test_xdata = np.array([[0,0],[0,1],[1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = AND_obj.predict(input_data)
    print(input_data, " = ", logical_val)

print('-----------------')
test_tdata = np.array([0,0,0,1])
accuracy_ret = AND_obj.accuracy(test_xdata, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  0
[1 0]  =  0
[1 1]  =  1
-----------------
Accuracy =>  1.0


In [16]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([0,1,1,1]) # 정답(target) 데이터
OR_obj = LogicGate("OR_GATE", xdata, tdata) # 모델 객체 생성
OR_obj.train() # 학습

initial loss value =  1.6231876134936636
step =  0 loss value =  1.620323849320995
step =  1000 loss value =  0.6729450745542465
step =  2000 loss value =  0.413229127532136
step =  3000 loss value =  0.2941024301109828
step =  4000 loss value =  0.2268030950242299
step =  5000 loss value =  0.1839253468511944
step =  6000 loss value =  0.1543657843539086
step =  7000 loss value =  0.13282012674953125
step =  8000 loss value =  0.11645161669703236


In [17]:
test_xdata = np.array([[0,0],[0,1],[1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = OR_obj.predict(input_data)
    print(input_data, " = ", logical_val)

print('-----------------')
test_tdata = np.array([0,1,1,1])
accuracy_ret = OR_obj.accuracy(test_xdata, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  1
-----------------
Accuracy =>  1.0


In [18]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([1,1,1,0]) # 정답(target) 데이터
NAND_obj = LogicGate("NAND_GATE", xdata, tdata) # 모델 객체 생성
NAND_obj.train() # 학습

initial loss value =  2.8641001880698105
step =  0 loss value =  2.858759026991561
step =  1000 loss value =  1.064490999359061
step =  2000 loss value =  0.6835263139567234
step =  3000 loss value =  0.5043064462573233
step =  4000 loss value =  0.39848008473195373
step =  5000 loss value =  0.3285776781056434
step =  6000 loss value =  0.2790482933149563
step =  7000 loss value =  0.2421874701643597
step =  8000 loss value =  0.21373077734770823


In [19]:
test_xdata = np.array([[0,0],[0,1],[1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = NAND_obj.predict(input_data)
    print(input_data, " = ", logical_val)

print('-----------------')
test_tdata = np.array([1,1,1,0])
accuracy_ret = NAND_obj.accuracy(test_xdata, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  1
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
-----------------
Accuracy =>  1.0


In [23]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([0,1,1,0]) # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata) # 모델 객체 생성
XOR_obj.train() # 학습

initial loss value =  3.4866305485393925
step =  0 loss value =  3.469275245075927
step =  1000 loss value =  2.7734140176481024
step =  2000 loss value =  2.77262229192657
step =  3000 loss value =  2.772589385563581
step =  4000 loss value =  2.772587984774808
step =  5000 loss value =  2.7725879249138528
step =  6000 loss value =  2.772587922354212
step =  7000 loss value =  2.7725879222447514
step =  8000 loss value =  2.7725879222400707
step =  9000 loss value =  2.77258792223987
step =  10000 loss value =  2.772587922239862
step =  11000 loss value =  2.7725879222398615
step =  12000 loss value =  2.7725879222398615
step =  13000 loss value =  2.7725879222398615
step =  14000 loss value =  2.7725879222398615
step =  15000 loss value =  2.7725879222398615
step =  16000 loss value =  2.7725879222398615
step =  17000 loss value =  2.7725879222398615
step =  18000 loss value =  2.7725879222398615
step =  19000 loss value =  2.7725879222398615
step =  20000 loss value =  2.77258792223

In [24]:
test_xdata = np.array([[0,0],[0,1],[1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = XOR_obj.predict(input_data)
    print(input_data, " = ", logical_val)

print('-----------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_xdata, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  0
[1 0]  =  0
[1 1]  =  1
-----------------
Accuracy =>  0.25


In [28]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([0,0,0,1]) # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata) # 모델 객체 생성
AND_obj.train() # 학습

initial loss value =  4.782322278392741
step =  0 loss value =  4.721398752893481
step =  1000 loss value =  1.0192655015080336
step =  2000 loss value =  0.6653908145761547
step =  3000 loss value =  0.4942793620159016
step =  4000 loss value =  0.39210536433457516
step =  5000 loss value =  0.3241761281666127
step =  6000 loss value =  0.2758338855466438
step =  7000 loss value =  0.23974166297513802
step =  8000 loss value =  0.21181022010414874
step =  9000 loss value =  0.18958032204097214
step =  10000 loss value =  0.1714855213632645
step =  11000 loss value =  0.15648179324670933
step =  12000 loss value =  0.14384721087665867
step =  13000 loss value =  0.1330670720160249
step =  14000 loss value =  0.12376480791248606
step =  15000 loss value =  0.11565873934513315
step =  16000 loss value =  0.1085340782431688
step =  17000 loss value =  0.10222426328123654
step =  18000 loss value =  0.09659819710306419
step =  19000 loss value =  0.09155132141996185
step =  20000 loss valu

In [29]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([0,1,1,1]) # 정답(target) 데이터
OR_obj = LogicGate("OR_GATE", xdata, tdata) # 모델 객체 생성
OR_obj.train() # 학습

initial loss value =  1.8632939623545788
step =  0 loss value =  1.8604059557851307
step =  1000 loss value =  0.7287389117146369
step =  2000 loss value =  0.435279190891464
step =  3000 loss value =  0.30556955870285657
step =  4000 loss value =  0.2337323089041811
step =  5000 loss value =  0.18853035055439865
step =  6000 loss value =  0.15763354658257506
step =  7000 loss value =  0.13525253306471233
step =  8000 loss value =  0.11832922917330782
step =  9000 loss value =  0.10510357662092962
step =  10000 loss value =  0.09449407768225433
step =  11000 loss value =  0.08580098940296875
step =  12000 loss value =  0.07855226161074032
step =  13000 loss value =  0.07241827219437598
step =  14000 loss value =  0.06716208914662856
step =  15000 loss value =  0.06260915265723681
step =  16000 loss value =  0.05862809224894173
step =  17000 loss value =  0.0551181937740546
step =  18000 loss value =  0.0520009796646097
step =  19000 loss value =  0.049214412530583417
step =  20000 loss

In [30]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습(train) 데이터
tdata = np.array([1,1,1,0]) # 정답(target) 데이터
NAND_obj = LogicGate("NAND_GATE", xdata, tdata) # 모델 객체 생성
NAND_obj.train() # 학습

initial loss value =  2.9136380969477353
step =  0 loss value =  2.9084521487529713
step =  1000 loss value =  1.0779733321794658
step =  2000 loss value =  0.6890900445952912
step =  3000 loss value =  0.5073654992309761
step =  4000 loss value =  0.4004156891170007
step =  5000 loss value =  0.3299097897460698
step =  6000 loss value =  0.2800188313654015
step =  7000 loss value =  0.24292462901383688
step =  8000 loss value =  0.21430882164399956
step =  9000 loss value =  0.1915914938494911
step =  10000 loss value =  0.17313770469101614
step =  11000 loss value =  0.15786222914456627
step =  12000 loss value =  0.1450171702247657
step =  13000 loss value =  0.13407081047758546
step =  14000 loss value =  0.12463506429368706
step =  15000 loss value =  0.11642023727961781
step =  16000 loss value =  0.1092058272380575
step =  17000 loss value =  0.10282111245781199
step =  18000 loss value =  0.0971319096352291
step =  19000 loss value =  0.09203133335350816
step =  20000 loss valu

In [32]:
input_data = np.array([[0,0],[0,1],[1,0],[1,1]])
s1 = [] # NAND 출력 저장
s2 = [] # OR 출력 저장
new_input_data = [] # AND 입력 저장
final_output = [] # 최종 출력 저장
for index in range(len(input_data)):
    s1 = NAND_obj.predict(input_data[index])
    s2 = OR_obj.predict(input_data[index])
    new_input_data.append(s1[-1])
    new_input_data.append(s2[-1])
    (sigmoid_val, logical_val) = AND_obj.predict(np.array(new_input_data))
    final_output.append(logical_val)
    new_input_data = []

for index in range(len(input_data)):
    print(input_data[index], " = ", final_output[index])

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0


# 딥러닝으로 XOR 해결

In [33]:
import numpy as np

def sigmoid(x): # 시그모이드 함수
    return 1. / (1. + np.exp(-x))

In [34]:
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [58]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.xdata = xdata.reshape(4,2) # 입력층 데이터
        self.tdata = tdata.reshape(4,1) # 정답 데이터

        self.W2 = np.random.rand(2,6) # 은닉층 가중치
        self.b2 = np.random.rand(6) # 은닉층 바이어스

        self.W3 = np.random.rand(6,1) # 출력층 가중치
        self.b3 = np.random.rand(1) # 출력층 바이어스

        self.learning_rate = 1e-2 # 학습률

    def loss_val(self):
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        y = a3 = sigmoid(z3)
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1-y)+delta))

    def train(self):
        f = lambda x : self.loss_val()
        print("Initial loss value = ", self.loss_val())
        for step in range(5001):
            self.W2 -= self.learning_rate * numerical_derivative(f,self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f,self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f,self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f,self.b3)
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ",self.loss_val()) # 학습중 손실값 출력
    
    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2,self.W3) + self.b3
        y = a3 = sigmoid(z3)
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result

    def accuracy(self, test_xdata, test_tdata):
        matched_list = [] # 정답과 일치한 내용 저장
        not_matched_list = [] # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index) # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index) # 정답과 일치한 내용 추가

        accuracy_val = len(matched_list) / len(test_xdata) # 정확도 계산
        return accuracy_val

In [65]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,0,0,1]) # 정답 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)
AND_obj.train()

Initial loss value =  8.496461238675856
step =  0 loss value =  8.181109455768656
step =  1000 loss value =  1.9386258669807255
step =  2000 loss value =  1.0512157177856951
step =  3000 loss value =  0.49572125693697805
step =  4000 loss value =  0.2652982332333883
step =  5000 loss value =  0.16636548787755268


In [66]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = AND_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,0,0,1])
accuracy_ret = AND_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  0
[1 0]  =  0
[1 1]  =  1
------------------
Accuracy =>  1.0


In [67]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,1,1,0]) # 정답 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train()

Initial loss value =  3.934744540738512
step =  0 loss value =  3.8622325825009476
step =  1000 loss value =  2.756955966863954
step =  2000 loss value =  2.7360203000252694
step =  3000 loss value =  2.690205621972233
step =  4000 loss value =  2.5847660394941285
step =  5000 loss value =  2.3751296164644904


In [68]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  1
------------------
Accuracy =>  0.75
